# TSA Chapter 12: Wavelet Analysis of US Real GDP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/TSA/blob/main/TSA_ch12/TSA_ch12_wavelet/TSA_ch12_wavelet.ipynb)

In [ ]:
!pip install matplotlib numpy scipy statsmodels PyWavelets -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pywt
from statsmodels.tsa.filters.hp_filter import hpfilter

In [ ]:
# Style configuration
COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#DC3545',
    'green':  '#2E7D32',
    'orange': '#E67E22',
    'gray':   '#666666',
    'purple': '#8E44AD',
}

plt.rcParams.update({
    'axes.facecolor':     'none',
    'figure.facecolor':   'none',
    'savefig.transparent': True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          False,
    'font.size':          9,
    'axes.labelsize':     9,
    'axes.titlesize':     10,
    'legend.fontsize':    8,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
})

In [ ]:
# Chart: ch12_gdp_scalogram
# Morlet CWT scalogram of US real GDP HP cycle
data = sm.datasets.macrodata.load_pandas().data
gdp = np.log(data['realgdp'].values)
cycle, _ = hpfilter(gdp, lamb=1600)

dt = 1
wavelet = 'cmor1.5-1.0'
scales = np.geomspace(2, 64, 80)
coeffs, freqs = pywt.cwt(cycle, scales, wavelet, sampling_period=dt)
periods = 1 / freqs
power = np.abs(coeffs)**2
T = len(cycle)
timeaxis = np.arange(T)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), gridspec_kw={'height_ratios': [1, 2.5]})
axes[0].plot(timeaxis, cycle, color=COLORS['blue'], lw=1.3)
axes[0].axhline(0, color='black', lw=0.6)
axes[0].set_title('HP-cycle of US Real GDP')
axes[0].set_ylabel('Cycle')
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('none')

im = axes[1].contourf(timeaxis, periods, power, levels=30, cmap='YlOrRd')
plt.colorbar(im, ax=axes[1], label='CWT Power')
axes[1].axhline(6,  color=COLORS['blue'], lw=1.2, linestyle='--', label='6q')
axes[1].axhline(32, color=COLORS['red'],  lw=1.2, linestyle='--', label='32q')
axes[1].set_ylim(4, 64)
axes[1].set_ylabel('Period (quarters)')
axes[1].set_xlabel('Quarter')
axes[1].set_title('Morlet CWT Scalogram — US Real GDP Cycle')
axes[1].legend(loc='upper right', frameon=False)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('ch12_gdp_scalogram.pdf', bbox_inches='tight', dpi=150)
plt.savefig('ch12_gdp_scalogram.png', bbox_inches='tight', dpi=150)
plt.show()